# 09 - Simulated Daily Orders Feed (job task 1)
Stands in for a real source system. Each run drops new files into the landing volume:
- `orders`: 3 new valid orders + 1 duplicate + 1 bad row (null order_id) -> tests Silver expectations every run
- `order_items`: 1 line item per new order -> new sales flow all the way to `gold.fact_orders`

In production this task would not exist; files would arrive from the source system.

In [0]:
import uuid
import pandas as pd
from datetime import datetime, timedelta
from pyspark.sql import functions as F

catalog = "retail_lakehouse"
raw = f"/Volumes/{catalog}/bronze/raw_files"
now = datetime.now()
fmt = "%Y-%m-%d %H:%M:%S"
stamp = now.strftime("%Y%m%d_%H%M%S")

order_cols = [
    "order_id", "customer_id", "order_status", "order_purchase_timestamp",
    "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "order_estimated_delivery_date",
]
item_cols = [
    "order_id", "order_item_id", "product_id", "seller_id",
    "shipping_limit_date", "price", "freight_value",
]

# Random real customers and real product/seller/price combos so all joins work
customers = [r.customer_id for r in
             spark.table(f"{catalog}.silver.customers").select("customer_id")
                  .orderBy(F.rand()).limit(3).collect()]
items = (spark.table(f"{catalog}.silver.order_items")
              .select("product_id", "seller_id", "price", "freight_value")
              .orderBy(F.rand()).limit(3).collect())

orders, order_items = [], []
for cid, item in zip(customers, items):
    oid = uuid.uuid4().hex
    orders.append({
        "order_id": oid,
        "customer_id": cid,
        "order_status": "approved",
        "order_purchase_timestamp": now.strftime(fmt),
        "order_approved_at": (now + timedelta(minutes=10)).strftime(fmt),
        "order_delivered_carrier_date": None,
        "order_delivered_customer_date": None,
        "order_estimated_delivery_date": (now + timedelta(days=10)).strftime(fmt),
    })
    order_items.append({
        "order_id": oid,
        "order_item_id": 1,
        "product_id": item.product_id,
        "seller_id": item.seller_id,
        "shipping_limit_date": (now + timedelta(days=2)).strftime(fmt),
        "price": float(item.price),
        "freight_value": float(item.freight_value),
    })

# Data quality traps: one duplicate + one bad row
existing = spark.table(f"{catalog}.bronze.orders_raw").select(*order_cols).limit(1).toPandas()
orders.append(existing.iloc[0].to_dict())
orders.append({**orders[0], "order_id": None})

pd.DataFrame(orders, columns=order_cols).to_csv(f"{raw}/orders/orders_batch_{stamp}.csv", index=False)
pd.DataFrame(order_items, columns=item_cols).to_csv(f"{raw}/order_items/order_items_batch_{stamp}.csv", index=False)

print(f"✅ Wrote {len(orders)} order rows and {len(order_items)} item rows (batch {stamp})")